# `Project Overview`

**Dataset Name:**
FER-2013 (Facial Expression Recognition 2013)

**Source:**
Available on Kaggle under *“Emotion Detection from Faces”* or *“FER-2013”* dataset.

**Description:**

* The dataset consists of **grayscale facial images**, each sized **48×48 pixels**.
* Each image contains a **single face**, labeled with one of several basic human emotions.
* It is widely used for training and testing deep learning models for **facial emotion recognition**.

**Emotion Classes (7 total):**

1. Angry
2. Disgust
3. Fear
4. Happy
5. Sad
6. Surprise
7. Neutral

# `Data Creation`

In [2]:
import os
import zipfile

zip_path = "Emotion Detection from Faces Dataset.zip"
extract_path = "Data"

if not os.path.exists(extract_path):
    print(f"Extracting dataset from '{zip_path}'...")
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Source zip file '{zip_path}' not found in the root directory.")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset extracted successfully!")
else:
    print(f"Dataset already extracted at '{extract_path}'")

Dataset already extracted at 'Data'


In [3]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

IMG_HEIGHT = 48
IMG_WIDTH = 48
IMG_CHANNELS = 3
BATCH_SIZE = 64
NUM_CLASSES = 7

# Setup image generators with standard [0, 1] rescaling
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    directory=os.path.join(extract_path, "train"),
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    directory=os.path.join(extract_path, "test"),
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Calculate class weights dynamically to deal with highly imbalanced classes
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = dict(enumerate(class_weights))

print("\nClass Weights Dictionary:")
class_labels = list(train_generator.class_indices.keys())
for i, label in enumerate(class_labels):
    print(f" - {label}: {class_weights_dict[i]:.4f}")

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.

Class Weights Dictionary:
 - angry: 1.0266
 - disgust: 9.4066
 - fear: 1.0010
 - happy: 0.5684
 - neutral: 0.8260
 - sad: 0.8491
 - surprise: 1.2934


# `Building the Model`

In [5]:
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, Add, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf

def residual_block(input_tensor, filters, stride=1):
    # First Conv
    x = Conv2D(filters, (3, 3), strides=stride, padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Second Conv
    x = Conv2D(filters, (3, 3), strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    
    # Shortcut path connection
    shortcut = input_tensor
    if stride != 1 or input_tensor.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same')(input_tensor)
        shortcut = BatchNormalization()(shortcut)
        
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def build_custom_resnet(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), num_classes=NUM_CLASSES):
    inputs = Input(shape=input_shape)
    
    # Entry convolution
    x = Conv2D(64, (3, 3), strides=1, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Residual Stages
    x = residual_block(x, 64, stride=1)
    x = residual_block(x, 64, stride=1)
    
    x = residual_block(x, 128, stride=2)  # Downsamples to 24x24
    x = residual_block(x, 128, stride=1)
    
    x = residual_block(x, 256, stride=2)  # Downsamples to 12x12
    x = residual_block(x, 256, stride=1)
    
    # Classification Head
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs, name="Custom_ResNet_48x48")
    return model

model = build_custom_resnet()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "Custom_ResNet_48x48"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 48, 48, 3)]  0           []                               
                                                                                                  
 conv2d_15 (Conv2D)             (None, 48, 48, 64)   1792        ['input_2[0][0]']                
                                                                                                  
 batch_normalization_16 (BatchN  (None, 48, 48, 64)  256         ['conv2d_15[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_13 (Activation)     (None, 48, 48, 64)   0           ['batch_normali

# `Training the Model`

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

EPOCHS = 30

early_stopper = EarlyStopping(
    monitor='val_accuracy',
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    x=train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stopper, reduce_lr],
    class_weight=class_weights_dict,
    verbose=1
)

Epoch 1/30
449/449 [==============================] - 240s 514ms/step - loss: 2.2986 - accuracy: 0.1506 - val_loss: 1.9316 - val_accuracy: 0.1531 - lr: 5.0000e-04
Epoch 2/30
449/449 [==============================] - 52s 116ms/step - loss: 2.0955 - accuracy: 0.1532 - val_loss: 1.9605 - val_accuracy: 0.1369 - lr: 5.0000e-04
Epoch 3/30
449/449 [==============================] - 52s 116ms/step - loss: 2.0165 - accuracy: 0.1568 - val_loss: 1.9597 - val_accuracy: 0.2275 - lr: 5.0000e-04
Epoch 4/30
449/449 [==============================] - 51s 114ms/step - loss: 1.9513 - accuracy: 0.1797 - val_loss: 1.8308 - val_accuracy: 0.2076 - lr: 5.0000e-04
Epoch 5/30
449/449 [==============================] - 51s 114ms/step - loss: 1.8652 - accuracy: 0.2012 - val_loss: 1.9357 - val_accuracy: 0.2639 - lr: 5.0000e-04
Epoch 6/30
449/449 [==============================] - 51s 114ms/step - loss: 1.8134 - accuracy: 0.2252 - val_loss: 1.9108 - val_accuracy: 0.1936 - lr: 5.0000e-04
Epoch 7/30
449/449 [=======

# `Saving the Model`

In [7]:
eval_metrics = model.evaluate(val_generator, verbose=1)
test_loss = eval_metrics[0]
test_acc = eval_metrics[1]

model.save("model.h5")

113/113 [==============================] - 4s 33ms/step - loss: 1.2871 - accuracy: 0.5525
